In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("heartzhacker/medical-imaging")

print("Path to dataset files:", path)

100%|██████████| 673M/673M [00:17<00:00, 39.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1


In [2]:
!pip install -q kagglehub timm albumentations torchmetrics grad-cam

import os, random, json, math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm  # gives us pretrained EfficientNet AND ViT from one library
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 43.1 MB/s eta 0:00:00
Device: cuda | GPU: Tesla T4


In [5]:
raw_path=path
print(raw_path)

# Inspect the structure first — Kaggle datasets vary in layout
for root, dirs, files in os.walk(raw_path):
    level = root.replace(raw_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:
            print(f"{indent}  {f}")
        if len(files) > 3:
            print(f"{indent}  ... ({len(files)} files)")

/root/.cache/kagglehub/datasets/heartzhacker/medical-imaging/versions/1
1/
  Medical-imaging-dataset/
    Peritonitis/
    Diverticulosis/
    Ureters/
    Neoplasm/


In [6]:
# Once you've confirmed the structure, build a flat list of (filepath, label)
CLASSES = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]  # adjust names if folder names differ
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

samples = []
for cls in CLASSES:
    cls_dir = None
    # search a couple levels deep for the folder matching this class name
    for root, dirs, files in os.walk(raw_path):
        if os.path.basename(root).lower() == cls.lower():
            cls_dir = root
            break
    if cls_dir is None:
        raise FileNotFoundError(f"Couldn't find folder for class '{cls}' — check Cell 2 output above")
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            samples.append((os.path.join(cls_dir, fname), class_to_idx[cls]))

print(f"Total images found: {len(samples)}")
from collections import Counter
print("Per class:", Counter([CLASSES[l] for _, l in samples]))

Total images found: 4000
Per class: Counter({'Diverticulosis': 1000, 'Neoplasm': 1000, 'Peritonitis': 1000, 'Ureters': 1000})


In [7]:
from sklearn.model_selection import train_test_split

paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.20, stratify=labels, random_state=SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

Train: 3200 | Val: 400 | Test: 400


# **Data pipeline: augmentation + MixUp/CutMix**

This is where the first real accuracy-boosting techniques go in. The paper only used basic flips/rotation/zoom/color jitter. We're adding:

Albumentations for more sophisticated, still medically-safe augmentation (elastic-ish transforms kept mild, CLAHE for contrast enhancement — useful on endoscopy images)
MixUp / CutMix at the batch level (implemented in the training loop, not here, but the dataset needs to return plain tensors for it to work)

In [8]:
IMG_SIZE = 448  # matches the paper

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.85, 1.0), ratio=(0.95, 1.05), p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.CLAHE(clip_limit=2.0, p=0.3),          # boosts local contrast — helps highlight mucosal texture
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.05, 0.1),
                     hole_width_range=(0.05, 0.1), p=0.3),  # random erasing — improves robustness
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class GIDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)["image"]
        label = self.labels[idx]
        return img, label

train_ds = GIDataset(train_paths, train_labels, train_transform)
val_ds   = GIDataset(val_paths, val_labels, eval_transform)
test_ds  = GIDataset(test_paths, test_labels, eval_transform)

BATCH_SIZE = 12  # matches the paper; bump to 16-24 if your Colab GPU has headroom (check with nvidia-smi)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Batches — train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

Batches — train: 266 val: 34 test: 34


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [10]:
class GastroHybridNet(nn.Module):
    """
    Dual-branch hybrid: EfficientNet-B4 (local/texture features) +
    ViT-Small (global/contextual features), fused before classification.
    """
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()

        # --- CNN branch: EfficientNet-B4, pretrained, classifier stripped ---
        self.cnn_backbone = timm.create_model(
            "efficientnet_b4", pretrained=True, num_classes=0, global_pool="avg"
        )
        cnn_feat_dim = self.cnn_backbone.num_features  # 1792 for EfficientNet-B4

        # --- ViT branch: ViT-Small, pretrained, classifier stripped ---
        self.vit_backbone = timm.create_model(
            "vit_small_patch16_224", pretrained=True, num_classes=0, global_pool="avg"
        )
        vit_feat_dim = self.vit_backbone.num_features  # 384 for ViT-Small

        # ViT-Small expects 224x224 inputs (its positional embeddings are fixed to that
        # patch grid) while EfficientNet-B4 can handle 448x448. We keep two input
        # resolutions and resize internally — see forward().
        self.vit_input_size = 224

        # --- Custom CNN refinement head (kept from the original paper's idea) ---
        # self.cnn_head = nn.Sequential(
        #     nn.Conv2d(self.cnn_backbone.feature_info[-1]["num_chs"], 512, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        #     nn.Conv2d(512, 256, kernel_size=3, padding=1),
        #     nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        #     nn.AdaptiveAvgPool2d(1),
        # )

         # --- Custom CNN refinement head (kept from the original paper's idea) ---
        self.cnn_head = nn.Sequential(
            nn.Conv2d(cnn_feat_dim, 512, kernel_size=3, padding=1),   # <-- fixed: was feature_info[-1]["num_chs"]
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        # cnn_head_out_dim = 256
        # Note: to route through cnn_head instead of the pooled backbone output,
        # we re-extract feature maps via forward_features (see forward()).
        cnn_head_out_dim = 256

        # --- Fusion + classifier ---
        fused_dim = cnn_head_out_dim + vit_feat_dim
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        # x arrives at 448x448 (matches the paper's resolution)

        # CNN branch: use forward_features to get the spatial map, then our custom head
        cnn_feat_map = self.cnn_backbone.forward_features(x)   # (B, C, H, W)
        cnn_feat = self.cnn_head(cnn_feat_map).flatten(1)      # (B, 256)

        # ViT branch: resize to 224x224 for ViT-Small's expected input
        x_vit = F.interpolate(x, size=(self.vit_input_size, self.vit_input_size),
                               mode="bilinear", align_corners=False)
        vit_feat = self.vit_backbone(x_vit)                    # (B, 384)

        # Fuse
        fused = torch.cat([cnn_feat, vit_feat], dim=1)
        fused = self.fusion(fused)
        out = self.classifier(fused)
        return out


# Quick sanity check — build the model and verify shapes before touching real data
model = GastroHybridNet(num_classes=4).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params/1e6:.1f}M | Trainable: {n_trainable/1e6:.1f}M")

dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print("Output shape:", out.shape)  # should be [2, 4]

Total params: 49.0M | Trainable: 49.0M
Output shape: torch.Size([2, 4])


In [11]:
def test_batch_memory(model, batch_size, img_size=IMG_SIZE, use_amp=True):
    model.train()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    dummy_x = torch.randn(batch_size, 3, img_size, img_size).to(device)
    dummy_y = torch.randint(0, 4, (batch_size,)).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    optimizer.zero_grad()
    with torch.cuda.amp.autocast(enabled=use_amp):
        out = model(dummy_x)
        loss = F.cross_entropy(out, dummy_y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Batch size {batch_size} | AMP={use_amp} | Peak memory: {peak_mem:.2f} GB / {total_mem:.2f} GB available")
    if peak_mem > total_mem * 0.85:
        print("⚠️  Cutting it close — consider lowering batch size or enabling gradient accumulation")
    else:
        print("✅ Should be safe for a full training run")

test_batch_memory(model, batch_size=BATCH_SIZE, use_amp=True)

/tmp/ipykernel_1269/608901064.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_1269/608901064.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Batch size 12 | AMP=True | Peak memory: 6.48 GB / 15.64 GB available
✅ Should be safe for a full training run


In [12]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/gastronet_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/gastronet_checkpoints


In [13]:
class MixupCutmix:
    """
    Randomly applies MixUp or CutMix to a batch. Both blend two training images
    together (and their labels), which is a well-established regularizer that
    reduces overfitting and often buys a genuine accuracy improvement on
    small-to-medium datasets like this one.
    """
    def __init__(self, mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, num_classes=4):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.num_classes = num_classes

    def _rand_bbox(self, size, lam):
        W, H = size[3], size[2]
        cut_rat = math.sqrt(1. - lam)
        cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
        cx, cy = np.random.randint(W), np.random.randint(H)
        x1 = np.clip(cx - cut_w // 2, 0, W)
        y1 = np.clip(cy - cut_h // 2, 0, H)
        x2 = np.clip(cx + cut_w // 2, 0, W)
        y2 = np.clip(cy + cut_h // 2, 0, H)
        return x1, y1, x2, y2

    def __call__(self, x, y):
        if np.random.rand() > self.prob:
            y_onehot = F.one_hot(y, self.num_classes).float()
            return x, y_onehot, y_onehot, 1.0  # no-op: lam=1 means "all original"

        y_onehot = F.one_hot(y, self.num_classes).float()
        perm = torch.randperm(x.size(0)).to(x.device)
        y_a, y_b = y_onehot, y_onehot[perm]

        if np.random.rand() < 0.5:  # MixUp
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            x = lam * x + (1 - lam) * x[perm]
        else:  # CutMix
            lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
            x1, y1, x2, y2 = self._rand_bbox(x.size(), lam)
            x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
            lam = 1 - ((x2 - x1) * (y2 - y1) / (x.size(-1) * x.size(-2)))

        return x, y_a, y_b, lam


def soft_ce_loss(logits, y_a, y_b, lam, smoothing=0.1):
    """Cross-entropy that works with soft/mixed labels + label smoothing baked in."""
    n_classes = logits.size(1)
    y_a_smooth = y_a * (1 - smoothing) + smoothing / n_classes
    y_b_smooth = y_b * (1 - smoothing) + smoothing / n_classes
    log_probs = F.log_softmax(logits, dim=1)
    loss_a = -(y_a_smooth * log_probs).sum(dim=1).mean()
    loss_b = -(y_b_smooth * log_probs).sum(dim=1).mean()
    return lam * loss_a + (1 - lam) * loss_b

mixup_cutmix = MixupCutmix(prob=0.5, num_classes=4)

In [14]:
EPOCHS = 50
WARMUP_EPOCHS = 3
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda')  # updated API, no more deprecation warning

def train_one_epoch(model, loader, optimizer, scaler, use_mixup=True):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        if use_mixup:
            x, y_a, y_b, lam = mixup_cutmix(x, y)
        else:
            y_onehot = F.one_hot(y, 4).float()
            y_a, y_b, lam = y_onehot, y_onehot, 1.0

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss = soft_ce_loss(logits, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()  # approximate — mixup makes "correct" fuzzy, but useful as a trend signal
        total += x.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss = F.cross_entropy(logits, y)
        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

In [17]:
!rm -rf /content/checkpoints

In [19]:
import shutil

LOCAL_CKPT_DIR = "/content/checkpoints"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/gastronet_checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

LOCAL_LATEST = os.path.join(LOCAL_CKPT_DIR, "latest_full.pt")
DRIVE_BEST_WEIGHTS = os.path.join(DRIVE_CKPT_DIR, "best_weights_only.pt")
DRIVE_BEST_META = os.path.join(DRIVE_CKPT_DIR, "best_meta.json")

FULL_CKPT_EVERY = 5  # save resumable checkpoint locally every N epochs

def load_checkpoint_if_exists(model, optimizer, scheduler, scaler, path):
    if os.path.exists(path):
        print(f"Found existing local checkpoint at {path} — resuming...")
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        return ckpt["epoch"] + 1, ckpt["best_val_acc"], ckpt.get("history", [])
    return 0, 0.0, []

start_epoch, best_val_acc, history = load_checkpoint_if_exists(
    model, optimizer, scheduler, scaler, LOCAL_LATEST
)

PATIENCE = 10
epochs_no_improve = 0

for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, use_mixup=True)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "lr": current_lr})

    # --- Full resumable checkpoint, local disk, every FULL_CKPT_EVERY epochs ---
    if (epoch + 1) % FULL_CKPT_EVERY == 0 or epoch == EPOCHS - 1:
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history,
        }, LOCAL_LATEST)
        print(f"  Local full checkpoint saved (epoch {epoch+1})")

    # --- Lightweight best-weights, Drive, only when val improves ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), DRIVE_BEST_WEIGHTS)
        with open(DRIVE_BEST_META, "w") as f:
            json.dump({"epoch": epoch, "val_acc": val_acc, "history": history}, f)
        print(f"   New best val acc: {best_val_acc:.4f} — weights saved to Drive")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"    Early stopping — no improvement for {PATIENCE} epochs")
            break

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

Epoch 1/50 | LR: 0.000194 | Train loss: 0.5886 acc: 0.8468 | Val loss: 0.1912 acc: 0.9775
   New best val acc: 0.9775 — weights saved to Drive
Epoch 2/50 | LR: 0.000192 | Train loss: 0.5970 acc: 0.8412 | Val loss: 0.1704 acc: 0.9775
Epoch 3/50 | LR: 0.000189 | Train loss: 0.5711 acc: 0.8293 | Val loss: 0.1779 acc: 0.9850
   New best val acc: 0.9850 — weights saved to Drive
Epoch 4/50 | LR: 0.000186 | Train loss: 0.5525 acc: 0.8703 | Val loss: 0.1533 acc: 0.9800
Epoch 5/50 | LR: 0.000182 | Train loss: 0.5367 acc: 0.8565 | Val loss: 0.1819 acc: 0.9825
  Local full checkpoint saved (epoch 5)
Epoch 6/50 | LR: 0.000178 | Train loss: 0.5377 acc: 0.8656 | Val loss: 0.1923 acc: 0.9775
Epoch 7/50 | LR: 0.000174 | Train loss: 0.5414 acc: 0.8575 | Val loss: 0.1722 acc: 0.9775
Epoch 8/50 | LR: 0.000170 | Train loss: 0.5660 acc: 0.8631 | Val loss: 0.1654 acc: 0.9800
Epoch 9/50 | LR: 0.000165 | Train loss: 0.5376 acc: 0.8640 | Val loss: 0.2124 acc: 0.9775
Epoch 10/50 | LR: 0.000159 | Train loss: 0.5

In [20]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

# Rebuild a fresh model instance and load the best weights (Drive-saved, weights-only)
eval_model = GastroHybridNet(num_classes=4).to(device)
eval_model.load_state_dict(torch.load(DRIVE_BEST_WEIGHTS, map_location=device))
eval_model.eval()

test_loss, test_acc, test_preds, test_labels = evaluate(eval_model, test_loader)

print(f"\n{'='*50}")
print(f"FINAL TEST SET RESULTS (n={len(test_labels)})")
print(f"{'='*50}")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average="macro"
)
print(f"\nMacro Precision: {precision*100:.2f}%")
print(f"Macro Recall:    {recall*100:.2f}%")
print(f"Macro F1:        {f1*100:.2f}%")

print(f"\n--- Per-class report ---")
print(classification_report(test_labels, test_preds, target_names=CLASSES, digits=4))

print(f"\n--- Confusion Matrix ---")
cm = confusion_matrix(test_labels, test_preds)
print("Rows = actual, Columns = predicted")
print("Classes:", CLASSES)
print(cm)


FINAL TEST SET RESULTS (n=400)
Test Accuracy: 98.50%
Test Loss: 0.1627

Macro Precision: 98.50%
Macro Recall:    98.50%
Macro F1:        98.50%

--- Per-class report ---
                precision    recall  f1-score   support

Diverticulosis     0.9901    1.0000    0.9950       100
      Neoplasm     0.9899    0.9800    0.9849       100
   Peritonitis     0.9900    0.9900    0.9900       100
       Ureters     0.9700    0.9700    0.9700       100

      accuracy                         0.9850       400
     macro avg     0.9850    0.9850    0.9850       400
  weighted avg     0.9850    0.9850    0.9850       400


--- Confusion Matrix ---
Rows = actual, Columns = predicted
Classes: ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']
[[100   0   0   0]
 [  0  98   0   2]
 [  0   0  99   1]
 [  1   1   1  97]]


The idea

Instead of predicting from a single forward pass, we run each test image through several augmented versions (flips, small rotations) and average the predictions. Misclassifications often happen on borderline/ambiguous images where one specific view confuses the model — averaging across views smooths that out. This is a well-established free accuracy boost, typically +0.25–1%.

In [21]:
import torch.nn.functional as F

def tta_predict(model, dataset, device, n_augments=8):
    """
    Runs each test image through the original + several flipped/rotated versions,
    averages softmax probabilities, and returns final predictions.
    """
    model.eval()

    # TTA transform variants — kept mild and medically-plausible, same spirit as training augmentation
    tta_transforms = [
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),  # original
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Rotate(limit=(10,10), p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Rotate(limit=(-10,-10), p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.CLAHE(clip_limit=2.0, p=1.0),
                    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    ][:n_augments]

    all_final_preds = []
    all_labels = []

    with torch.no_grad():
        for idx in range(len(dataset.paths)):
            img_bgr = cv2.imread(dataset.paths[idx])
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            label = dataset.labels[idx]

            probs_sum = torch.zeros(1, len(CLASSES)).to(device)
            for tfm in tta_transforms:
                x = tfm(image=img_rgb)["image"].unsqueeze(0).to(device)
                with torch.amp.autocast('cuda'):
                    logits = model(x)
                    probs = F.softmax(logits, dim=1)
                probs_sum += probs

            probs_avg = probs_sum / len(tta_transforms)
            final_pred = probs_avg.argmax(dim=1).item()
            all_final_preds.append(final_pred)
            all_labels.append(label)

            if (idx + 1) % 100 == 0:
                print(f"  Processed {idx+1}/{len(dataset.paths)}")

    return all_final_preds, all_labels

print("Running TTA on test set (8 augmented views per image)...")
tta_preds, tta_labels = tta_predict(eval_model, test_ds, device, n_augments=8)

tta_acc = np.mean(np.array(tta_preds) == np.array(tta_labels))
print(f"\n{'='*50}")
print(f"TTA TEST ACCURACY: {tta_acc*100:.2f}%")
print(f"(vs single-pass test accuracy: 98.50%)")
print(f"{'='*50}")

print("\n--- TTA Per-class report ---")
print(classification_report(tta_labels, tta_preds, target_names=CLASSES, digits=4))

print("\n--- TTA Confusion Matrix ---")
cm_tta = confusion_matrix(tta_labels, tta_preds)
print("Classes:", CLASSES)
print(cm_tta)

Running TTA on test set (8 augmented views per image)...
  Processed 100/400
  Processed 200/400
  Processed 300/400
  Processed 400/400

TTA TEST ACCURACY: 98.75%
(vs single-pass test accuracy: 98.50%)

--- TTA Per-class report ---
                precision    recall  f1-score   support

Diverticulosis     0.9900    0.9900    0.9900       100
      Neoplasm     0.9802    0.9900    0.9851       100
   Peritonitis     0.9901    1.0000    0.9950       100
       Ureters     0.9898    0.9700    0.9798       100

      accuracy                         0.9875       400
     macro avg     0.9875    0.9875    0.9875       400
  weighted avg     0.9875    0.9875    0.9875       400


--- TTA Confusion Matrix ---
Classes: ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']
[[ 99   1   0   0]
 [  0  99   0   1]
 [  0   0 100   0]
 [  1   1   1  97]]


In [22]:
# Load second model snapshot from local full checkpoint
model_b = GastroHybridNet(num_classes=4).to(device)
ckpt = torch.load(LOCAL_LATEST, map_location=device)
model_b.load_state_dict(ckpt["model_state"])
model_b.eval()
print(f"Loaded second snapshot from epoch {ckpt['epoch']+1}, its own best_val_acc recorded: {ckpt['best_val_acc']:.4f}")

def tta_probs(model, dataset, device, n_augments=8):
    """Same as tta_predict but returns averaged PROBABILITIES, not final predictions —
    needed so we can combine across models before taking argmax."""
    model.eval()
    tta_transforms = [
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Rotate(limit=(10,10), p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Rotate(limit=(-10,-10), p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.CLAHE(clip_limit=2.0, p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    ][:n_augments]

    all_probs = []
    with torch.no_grad():
        for idx in range(len(dataset.paths)):
            img_bgr = cv2.imread(dataset.paths[idx])
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            probs_sum = torch.zeros(1, len(CLASSES)).to(device)
            for tfm in tta_transforms:
                x = tfm(image=img_rgb)["image"].unsqueeze(0).to(device)
                with torch.amp.autocast('cuda'):
                    logits = model(x)
                    probs_sum += F.softmax(logits, dim=1)
            all_probs.append((probs_sum / len(tta_transforms)).cpu())
    return torch.cat(all_probs, dim=0)  # (N, num_classes)

print("Running TTA for model A (best, epoch 3)...")
probs_a = tta_probs(eval_model, test_ds, device, n_augments=8)

print("Running TTA for model B (later snapshot)...")
probs_b = tta_probs(model_b, test_ds, device, n_augments=8)

# Simple average ensemble
probs_ensemble = (probs_a + probs_b) / 2
ensemble_preds = probs_ensemble.argmax(dim=1).numpy()
true_labels = np.array(test_ds.labels)

ensemble_acc = np.mean(ensemble_preds == true_labels)
print(f"\n{'='*50}")
print(f"CHECKPOINT ENSEMBLE + TTA TEST ACCURACY: {ensemble_acc*100:.2f}%")
print(f"(single best model + TTA: 98.75% | this run's raw single-pass: 98.50%)")
print(f"{'='*50}")

print("\n--- Ensemble Per-class report ---")
print(classification_report(true_labels, ensemble_preds, target_names=CLASSES, digits=4))
print("\n--- Ensemble Confusion Matrix ---")
print(confusion_matrix(true_labels, ensemble_preds))

Loaded second snapshot from epoch 10, its own best_val_acc recorded: 0.9850
Running TTA for model A (best, epoch 3)...
Running TTA for model B (later snapshot)...

CHECKPOINT ENSEMBLE + TTA TEST ACCURACY: 98.50%
(single best model + TTA: 98.75% | this run's raw single-pass: 98.50%)

--- Ensemble Per-class report ---
                precision    recall  f1-score   support

Diverticulosis     0.9900    0.9900    0.9900       100
      Neoplasm     0.9802    0.9900    0.9851       100
   Peritonitis     0.9900    0.9900    0.9900       100
       Ureters     0.9798    0.9700    0.9749       100

      accuracy                         0.9850       400
     macro avg     0.9850    0.9850    0.9850       400
  weighted avg     0.9850    0.9850    0.9850       400


--- Ensemble Confusion Matrix ---
[[99  1  0  0]
 [ 0 99  0  1]
 [ 0  0 99  1]
 [ 1  1  1 97]]


In [16]:
def save_checkpoint(state, filename):
    torch.save(state, filename)

def load_checkpoint_if_exists(model, optimizer, scheduler, scaler, path):
    if os.path.exists(path):
        print(f"Found existing checkpoint at {path} — resuming...")
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        return ckpt["epoch"] + 1, ckpt["best_val_acc"], ckpt.get("history", [])
    return 0, 0.0, []

LATEST_CKPT = os.path.join(CHECKPOINT_DIR, "latest.pt")
BEST_CKPT = os.path.join(CHECKPOINT_DIR, "best.pt")

start_epoch, best_val_acc, history = load_checkpoint_if_exists(
    model, optimizer, scheduler, scaler, LATEST_CKPT
)

PATIENCE = 10  # early stopping: stop if val acc doesn't improve for this many epochs
epochs_no_improve = 0

for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, use_mixup=True)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "lr": current_lr})

    # Always save "latest" so we can resume after any disconnect
    save_checkpoint({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_acc": best_val_acc,
        "history": history,
    }, LATEST_CKPT)

    # Separately save "best" whenever val accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        save_checkpoint({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history,
        }, BEST_CKPT)
        print(f"   New best val acc: {best_val_acc:.4f} — saved to {BEST_CKPT}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"    Early stopping — no improvement for {PATIENCE} epochs")
            break

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

Epoch 1/50 | LR: 0.000133 | Train loss: 0.6751 acc: 0.7982 | Val loss: 0.1873 acc: 0.9675
   New best val acc: 0.9675 — saved to /content/drive/MyDrive/gastronet_checkpoints/best.pt
Epoch 2/50 | LR: 0.000200 | Train loss: 0.6758 acc: 0.8055 | Val loss: 0.2172 acc: 0.9775
   New best val acc: 0.9775 — saved to /content/drive/MyDrive/gastronet_checkpoints/best.pt
Epoch 3/50 | LR: 0.000200 | Train loss: 0.6571 acc: 0.8120 | Val loss: 0.1943 acc: 0.9625
Epoch 4/50 | LR: 0.000200 | Train loss: 0.6267 acc: 0.8343 | Val loss: 0.1976 acc: 0.9600
Epoch 5/50 | LR: 0.000199 | Train loss: 0.6342 acc: 0.8255 | Val loss: 0.2043 acc: 0.9775
Epoch 6/50 | LR: 0.000198 | Train loss: 0.6006 acc: 0.8064 | Val loss: 0.1601 acc: 0.9750


KeyboardInterrupt: 